svd로 학습 -> 평가 -> 추론 하는 코드 개발.    
정리해서 pipeline class 만들기

In [2]:
import sys
import os

# 주피터 노트북 환경에서 __file__이 없으므로, 현재 워킹 디렉토리 기준으로 설정
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import logging
from pathlib import Path
import yaml

model_config_path = '/Users/visuworks/Desktop/movie_recommendation/modeling/models/config.yaml'
with open(model_config_path, 'r', encoding='utf-8') as f:
    model_config_dict = yaml.safe_load(f)

data_config_path = '/Users/visuworks/Desktop/movie_recommendation/modeling/utils/data_config.yaml'
with open(data_config_path, 'r', encoding='utf-8') as f:
    data_config_dict = yaml.safe_load(f)
    
print(data_config_dict)
print(model_config_dict)

{'data': {'min_user_ratings': 10, 'min_movie_ratings': 30}}
{'svd': {'n_factors': 20, 'n_epochs': 8, 'lr_all': 0.005, 'reg_all': 0.02, 'random_state': 42, 'verbose': True, 'test_size': 0.2, 'rating_scale': [0.5, 5.0], 'use_integrated_data': True}, 'item_based': {'top_k': 500, 'verbose': True, 'use_integrated_data': True}, 'ner': {'model_name': 'Qwen/Qwen2.5-3B-Instruct', 'allowed_models': ['Qwen/Qwen2.5-1.5B-Instruct', 'Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-7B-Instruct'], 'max_new_tokens': 128, 'temperature': 0.0, 'top_p': 0.9, 'do_sample': False, 'mps_single_device': True, 'system_prompt': 'You are a specialized information extraction system for movie queries. \nYour ONLY task is to extract structured information from the user\'s query and return it as JSON.\n\nExtract the following information from the user query:\n- actors: Array of actor names mentioned (배우 이름)\n- genres: Array of genres mentioned\n  \n  CRITICAL for genres: Extract genres ONLY from the predefined list below. \n

In [4]:
from modeling.models.svd.dataloader import load_train_test_df

df_train, df_test = load_train_test_df(data_config_dict, refresh=False)


✅ 캐시에서 Train/Test 데이터 로드: cache_train_70e1b2a1d2b3.csv


In [5]:
from surprise import SVD, Dataset, Reader
from surprise import accuracy

from modeling.utils.train import optimize_dataframe_for_surprise

# DataFrame 최적화 (trainset 생성 속도 향상)
print("데이터 최적화 중...")
df_train_opt = optimize_dataframe_for_surprise(df_train[['user_id', 'movie_id', 'rating']])
df_test_opt = optimize_dataframe_for_surprise(df_test[['user_id', 'movie_id', 'rating']])
print("✅ 최적화 완료\n")

# Reader 객체 생성 (평점 범위 지정)
reader = Reader(rating_scale=(0.5, 5.0))

train_data = Dataset.load_from_df(df_train_opt, reader)
trainset = train_data.build_full_trainset()
testset = [tuple(x) for x in df_test_opt[['user_id', 'movie_id', 'rating']].values]

데이터 최적화 중...
✅ 최적화 완료



In [6]:
svd_config = model_config_dict['svd']

In [7]:
algo = SVD(n_factors=svd_config['n_factors'], 
           n_epochs=svd_config['n_epochs'], 
           lr_all=svd_config['lr_all'], 
           reg_all=svd_config['reg_all'], 
           random_state=42, 
           verbose=True)
# algo.fit(trainset)

In [17]:
# 5. testset 중 일부분 샘플링해서 평가
import random

sample_size = 10000  # 원하는 샘플 크기 지정
if len(testset) > sample_size:
    sampled_testset = random.sample(testset, sample_size)
else:
    sampled_testset = testset

print(f"테스트셋에서 {len(sampled_testset)}개 샘플 추출하여 평가합니다.")

predictions = algo.test(sampled_testset)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print(f"샘플 테스트셋 RMSE: {rmse:.4f}, MAE: {mae:.4f}")

테스트셋에서 10000개 샘플 추출하여 평가합니다.
RMSE: 0.8139
MAE:  0.6133
샘플 테스트셋 RMSE: 0.8139, MAE: 0.6133


In [13]:
# import joblib

# # SVD 모델 저장
# svd_model_path = "/Users/visuworks/Desktop/movie_recommendation/modeling/models/data/svd_model.joblib"
# joblib.dump(algo, svd_model_path, compress=3)
# print(f"SVD 모델이 저장되었습니다: {svd_model_path}")

모델이 커지니까 저장하고 로드하는데 시간이 너무 오래걸린다는 이슈가 있네.... 흠
- 저장하는데 10분 안밖

In [18]:
import numpy as np

# 예시: Surprise SVD 모델에서 파라미터 추출
np.savez_compressed(
    "/Users/visuworks/Desktop/movie_recommendation/modeling/models/data/svd_params.npz",        # 저장할 파일 이름
    pu=algo.pu,              # 사용자 잠재벡터
    qi=algo.qi,              # 아이템 잠재벡터
    bu=algo.bu,              # 사용자 bias
    bi=algo.bi,              # 아이템 bias
    global_mean=algo.trainset.global_mean  # 전역 평균
)

In [1]:
import joblib

# 저장
# joblib.dump(trainset, "/Users/visuworks/Desktop/movie_recommendation/modeling/models/data/trainset.joblib")

In [ ]:
from surprise import SVD, Dataset, Reader
import numpy as np
# import pickle


# 2️⃣ 모델 객체 생성
model = SVD(n_factors=svd_config['n_factors'], 
           n_epochs=svd_config['n_epochs'], 
           lr_all=svd_config['lr_all'], 
           reg_all=svd_config['reg_all'], 
           random_state=42, 
           verbose=True)
model.trainset = trainset  # trainset을 수동으로 주입

# 3️⃣ 파라미터 로드
data = np.load("/Users/visuworks/Desktop/movie_recommendation/modeling/models/data/svd_params.npz")
model.pu = data["pu"]
model.qi = data["qi"]
model.bu = data["bu"]
model.bi = data["bi"]
model.global_mean = data["global_mean"].item()

In [12]:
model

In [13]:
model.predict(uid='123', iid='456')

Prediction(uid='123', iid='456', r_ui=None, est=3.53945932172165, details={'was_impossible': False})